# EfficientDet（PASCAL VOC 2007を用いた物体検出）

---
## 目的
物体検出モデルEfficientDetを，PASCAL VOC 2007データセットを用いて実装する．EfficientDetは，複数スケールの特徴マップを双方向かつ重み付きで融合する**BiFPN**（Bidirectional Feature Pyramid Network）を導入することで，少ない計算量で高精度な特徴ピラミッドを構築する手法である．本ノートブックでは，BiFPNの「学習可能な重み付き特徴融合」を中心に実装し，その効果を確認する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCDetection
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.ops import box_iou, box_convert, nms
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．VOCデータセットは，人物・乗り物・動物・室内物品など20クラスの物体に対して矩形領域（Bounding Box）が付与されたデータセットで，学習用（trainval）5011枚，評価用（test）4952枚の画像から構成されます．データセットの読み込みやDataLoaderの基本的な使い方については`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCDetection`で読み込みます．EfficientDetは原論文でも比較的大きな入力解像度を使用するため，本ノートブックでも`IMG_SIZE=512`とします（EfficientDet-D0と同じ解像度）．

In [ ]:
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
# 背景（物体なし）を0番として，クラスラベルは1〜20を割り当てる（GTボックスのラベル表現としてssd.ipynbと共通）
CLASS_TO_IDX = {name: i + 1 for i, name in enumerate(VOC_CLASSES)}
n_class = len(VOC_CLASSES)  # 分類ヘッドの出力クラス数（背景クラスは持たない．理由は後述）
IMG_SIZE = 512

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)


def parse_voc_target(target):
    """VOCDetectionが返すXML由来のdictを，(boxes, labels)のTensorに変換する"""
    objects = target['annotation']['object']
    if isinstance(objects, dict):  # 物体が1つだけの場合はdictのまま返るため，リストに揃える
        objects = [objects]

    boxes, labels = [], []
    for obj in objects:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])

    return {'boxes': torch.tensor(boxes, dtype=torch.float32), 'labels': torch.tensor(labels, dtype=torch.long)}


class VOCDetectionDataset(torch.utils.data.Dataset):
    """VOCDetectionをラップし，画像のリサイズに合わせてボックス座標をスケールしたTensorを返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCDetection(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.to_tensor = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, target = self.voc[idx]
        w, h = image.size
        target = parse_voc_target(target)

        # 元画像サイズ -> (img_size, img_size) へのリサイズに合わせてボックス座標をスケール
        scale = torch.tensor([self.img_size / w, self.img_size / h, self.img_size / w, self.img_size / h])
        target['boxes'] = target['boxes'] * scale

        image = self.to_tensor(image)
        return image, target


def collate_fn(batch):
    # 画像1枚あたりの物体数が異なるため，ボックス・ラベルはリストのまま扱う
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


train_data = VOCDetectionDataset('trainval')
test_data = VOCDetectionDataset('test')

print(len(train_data), len(test_data))
print('クラス数（背景を除く）:', n_class)

## バックボーン（EfficientNet-B0）
EfficientDetは，モデル名の通りバックボーンに**EfficientNet**を用いることが設計上の中心的な要素である．EfficientNetは，ネットワークの深さ・幅・入力解像度をバランスよく同時にスケールアップする「複合スケーリング（Compound Scaling）」により設計された効率の良い分類ネットワークであり，EfficientDetはこのバックボーンとBiFPN・検出ヘッドの規模を対応させてスケールさせる（D0〜D7）．

本シリーズの他の検出ノートブック（`ssd.ipynb`など）では事前学習済みバックボーンとしてResNet50を基本方針としているが，**EfficientDetについてはこの一般方針の例外として，事前学習済みEfficientNet-B0を使用する**．EfficientDet-D0が実際に採用しているバックボーンそのものであり，モデル名・設計思想と整合させるための意図的な例外である．

`efficientnet_b0`の`features`（`nn.Sequential`）に512×512の画像を入力すると，各ステージの出力は以下のような解像度・チャンネル数になる（事前に確認済み）．

| `features`のインデックス | 出力サイズ | チャンネル数 | ストライド |
|---|---|---|---|
| 3 | 64×64 | 40 | 8 |
| 5 | 32×32 | 112 | 16 |
| 7 | 16×16 | 320 | 32 |

この3段階の出力を，SSDや他の検出ノートブックにおけるC3・C4・C5と同様に，ストライド8・16・32の特徴マップ（P3・P4・P5の元になる特徴）として使用する．

In [ ]:
class EfficientDetBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        eff = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        # features[0:4]->C3(stride8,ch40), [4:6]->C4(stride16,ch112), [6:8]->C5(stride32,ch320)
        self.stem_to_c3 = eff.features[0:4]
        self.c3_to_c4 = eff.features[4:6]
        self.c4_to_c5 = eff.features[6:8]
        self.out_channels = [40, 112, 320]

    def forward(self, x):
        c3 = self.stem_to_c3(x)
        c4 = self.c3_to_c4(c3)
        c5 = self.c4_to_c5(c4)
        return c3, c4, c5


# 出力形状の確認
_backbone_check = EfficientDetBackbone().to(device)
_dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
for _f in _backbone_check(_dummy):
    print(_f.shape)
del _backbone_check, _dummy

## BiFPN（Bidirectional Feature Pyramid Network）
EfficientDetの最大の特徴は，バックボーンではなく**BiFPN**というneck構造にある．従来のFPN（Feature Pyramid Network）は，深い特徴マップから浅い特徴マップへ一方向（トップダウン）に情報を伝播するだけだったが，BiFPNは次の2点を導入する．

1. **双方向の情報伝播**：トップダウン（深い→浅い）に加えて，ボトムアップ（浅い→深い）の経路も設け，1つのBiFPNブロックの中で両方向に情報を伝播する．
2. **重み付き特徴融合（Weighted Feature Fusion）**：複数の特徴マップを足し合わせる際，単純な加算や平均ではなく，**学習可能なスカラー重み**を各入力に付与し，どの入力をどれだけ信頼するかをネットワーク自身に学習させる．具体的には，各重み$w_i$を`F.relu`に通して非負にしたのち正規化する

$$\text{fused} = \frac{\sum_i w_i \cdot x_i}{\sum_i w_i + \epsilon}$$

という「Fast Normalized Fusion」を用いる．Softmaxを使った正規化よりも計算コストが低く，論文ではSoftmax版とほぼ同等の精度が得られると報告されている．

本ノートブックでは，バックボーンから得られる3段階の特徴（P3・P4・P5）に対してBiFPNブロックを構成する．入力・出力とも解像度・チャンネル数が同じになるようにし，これを複数回（本ノートブックでは3回）繰り返して積み重ねることで，最終的な多スケール特徴ピラミッドを得る．

- **トップダウン経路**：最も深いP5から開始し，1つ上の解像度のP4と（アップサンプリングした）P5を重み付き融合する．同様にP4とP3を融合する．
- **ボトムアップ経路**：トップダウン経路の出力の中で最も浅いP3から開始し，1つ下の解像度に向けて（ダウンサンプリングしながら）再度重み付き融合する．中間レベル（P4）のボトムアップ融合では，「元の入力P4」「トップダウン経路で得たP4」「ダウンサンプリングしたP3の出力」という**3つの入力**を重み付き融合する点が特徴的である．

各融合の後には，パラメータ数と計算量を抑えるためのDepthwise Separable Convolution（`mobilenet_v1.ipynb`で扱った，チャンネルごとの畳み込みとポイントごとの畳み込みに分解する手法）を適用する．

In [ ]:
class SeparableConvBlock(nn.Module):
    """Depthwise Separable Conv + BatchNorm + 活性化関数（mobilenet_v1.ipynbと同様の分解）"""
    def __init__(self, channels):
        super().__init__()
        self.depthwise = nn.Conv2d(channels, channels, kernel_size=3, padding=1, groups=channels, bias=False)
        self.pointwise = nn.Conv2d(channels, channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(channels)
        self.act = nn.SiLU(inplace=True)  # EfficientNet系列に合わせてSiLU（Swish）を使用

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.act(x)


class WeightedFusion(nn.Module):
    """複数の特徴マップを，学習可能な非負スカラー重みで正規化しながら融合する（Fast Normalized Fusion）"""
    def __init__(self, num_inputs, channels, eps=1e-4):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(num_inputs))  # 学習可能な融合重み
        self.eps = eps
        self.conv = SeparableConvBlock(channels)

    def forward(self, *feats):
        w = F.relu(self.weights)  # 非負化
        w = w / (w.sum() + self.eps)  # 正規化（重みの合計が1になるようにする）
        fused = sum(wi * fi for wi, fi in zip(w, feats))
        return self.conv(fused)


class BiFPNBlock(nn.Module):
    """P3・P4・P5に対してトップダウン->ボトムアップの重み付き融合を1回行うブロック"""
    def __init__(self, channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')
        self.downsample = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.fuse_p4_td = WeightedFusion(2, channels)  # トップダウン: P4
        self.fuse_p3_out = WeightedFusion(2, channels)  # トップダウン: P3（最終出力を兼ねる）
        self.fuse_p4_out = WeightedFusion(3, channels)  # ボトムアップ: P4（3入力）
        self.fuse_p5_out = WeightedFusion(2, channels)  # ボトムアップ: P5

    def forward(self, p3, p4, p5):
        # トップダウン経路（深い -> 浅い）
        p5_td = p5
        p4_td = self.fuse_p4_td(p4, self.upsample(p5_td))
        p3_out = self.fuse_p3_out(p3, self.upsample(p4_td))

        # ボトムアップ経路（浅い -> 深い）。P4は3入力（元のP4，トップダウンのP4，ダウンサンプルしたP3）を融合する
        p4_out = self.fuse_p4_out(p4, p4_td, self.downsample(p3_out))
        p5_out = self.fuse_p5_out(p5_td, self.downsample(p4_out))

        return p3_out, p4_out, p5_out


class BiFPN(nn.Module):
    def __init__(self, in_channels_list, channels=64, num_repeats=3):
        super().__init__()
        # バックボーンの出力チャンネル数を，BiFPN内部の共通チャンネル数に揃える1x1畳み込み
        self.laterals = nn.ModuleList([nn.Conv2d(c, channels, kernel_size=1) for c in in_channels_list])
        self.blocks = nn.ModuleList([BiFPNBlock(channels) for _ in range(num_repeats)])

    def forward(self, c3, c4, c5):
        p3 = self.laterals[0](c3)
        p4 = self.laterals[1](c4)
        p5 = self.laterals[2](c5)
        for block in self.blocks:
            p3, p4, p5 = block(p3, p4, p5)
        return p3, p4, p5


# 出力形状の確認（P3・P4・P5とも同じチャンネル数64になる）
_bifpn_check = BiFPN([40, 112, 320], channels=64, num_repeats=3).to(device)
_c3 = torch.randn(1, 40, 64, 64).to(device)
_c4 = torch.randn(1, 112, 32, 32).to(device)
_c5 = torch.randn(1, 320, 16, 16).to(device)
for _f in _bifpn_check(_c3, _c4, _c5):
    print(_f.shape)
del _bifpn_check, _c3, _c4, _c5

## 検出ヘッド
BiFPNが出力する各レベル（P3・P4・P5）の特徴マップに対して，「クラス分類」と「ボックス回帰」を行う2つのサブネットを適用する．RetinaNet系の検出器と同様に，**畳み込みの重みは全てのピラミッドレベルで共有**する（レベルごとに別々の重みを持たせるとパラメータ数が増えるうえ，レベル間でスケールの違いに頑健な特徴を学習しやすくする狙いがある）．各レベルのセルには，複数スケール・複数アスペクト比のアンカーボックスを割り当てるため，出力チャンネル数はアンカー数に比例する．

- クラス分類サブネット：Depthwise Separable Convを3層重ねたのち，`num_anchors × n_class`チャンネルを出力する畳み込み層．
- ボックス回帰サブネット：同様の構成で，`num_anchors × 4`チャンネルを出力する．

In [ ]:
class DetectionHead(nn.Module):
    def __init__(self, channels, num_anchors, n_class, num_layers=3):
        super().__init__()
        self.cls_convs = nn.ModuleList([SeparableConvBlock(channels) for _ in range(num_layers)])
        self.box_convs = nn.ModuleList([SeparableConvBlock(channels) for _ in range(num_layers)])
        self.cls_out = nn.Conv2d(channels, num_anchors * n_class, kernel_size=3, padding=1)
        self.box_out = nn.Conv2d(channels, num_anchors * 4, kernel_size=3, padding=1)
        self.n_class = n_class

    def forward(self, features):
        cls_list, box_list = [], []
        for feat in features:
            cls_feat = feat
            for conv in self.cls_convs:
                cls_feat = conv(cls_feat)
            box_feat = feat
            for conv in self.box_convs:
                box_feat = conv(box_feat)

            cls_map = self.cls_out(cls_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, self.n_class)
            box_map = self.box_out(box_feat).permute(0, 2, 3, 1).contiguous().view(feat.size(0), -1, 4)
            cls_list.append(cls_map)
            box_list.append(box_map)

        return torch.cat(cls_list, dim=1), torch.cat(box_list, dim=1)

## アンカーボックスの生成
`ssd.ipynb`のデフォルトボックスと同様の発想で，P3・P4・P5（ストライド8・16・32）の各セルに対して，複数スケール・複数アスペクト比のアンカーボックスを配置する．EfficientDet（RetinaNetと同系統）の設計にならい，各レベルで

- スケール：$\{2^0, 2^{1/3}, 2^{2/3}\}$（基準サイズに対する倍率）
- アスペクト比：$\{0.5, 1, 2\}$

の組み合わせ，合計9種類のアンカーを各セルに配置する．基準サイズはストライドの4倍とする．座標は画像上のピクセル座標（中心座標・幅・高さ）で表す．

In [ ]:
ANCHOR_STRIDES = [8, 16, 32]
ANCHOR_SCALES = [2 ** 0, 2 ** (1 / 3), 2 ** (2 / 3)]
ANCHOR_RATIOS = [0.5, 1.0, 2.0]
num_anchors_per_cell = len(ANCHOR_SCALES) * len(ANCHOR_RATIOS)  # 9


def generate_anchors():
    anchors = []
    for stride in ANCHOR_STRIDES:
        fmap_size = IMG_SIZE // stride
        base_size = stride * 4.0
        for i in range(fmap_size):
            for j in range(fmap_size):
                cx = (j + 0.5) * stride
                cy = (i + 0.5) * stride
                for scale in ANCHOR_SCALES:
                    for ratio in ANCHOR_RATIOS:
                        w = base_size * scale * (ratio ** 0.5)
                        h = base_size * scale / (ratio ** 0.5)
                        anchors.append([cx, cy, w, h])
    return torch.tensor(anchors, dtype=torch.float32)


anchors = generate_anchors().to(device)  # (num_anchors, 4), (cx, cy, w, h) のピクセル座標
anchors_xyxy = box_convert(anchors, 'cxcywh', 'xyxy').clamp(0, IMG_SIZE)
print('アンカーの総数:', anchors.shape[0])

## ボックスのエンコード・デコードとアンカーのマッチング
`ssd.ipynb`と同様に，学習時にはGTボックスをマッチしたアンカーからの相対的なオフセットにエンコードして回帰対象とし，推論時にはオフセットをアンカーに適用して画像上の座標にデコードする．

マッチングも`ssd.ipynb`と同様に，各GTボックスに対してIoUが最大のアンカーを必ず正例にしたのち，IoUが閾値以上の残りのアンカーも正例にする．本ノートブックではFocal Lossを用いる都合上，RetinaNetの原論文にならい

- IoU $\geq 0.5$ ：正例（対応するクラス）
- IoU $< 0.4$ ：負例（背景）
- $0.4 \leq$ IoU $< 0.5$ ：**学習から除外**（どちらに割り当てるべきか曖昧なアンカー）

という3段階の閾値を設ける．除外されたアンカーはラベル`-1`として損失計算から除く．

In [ ]:
def encode_boxes(matched_boxes_xyxy, variances=(0.1, 0.2)):
    """マッチしたGTボックス（xyxy, ピクセル座標）を，アンカーからのオフセットにエンコードする"""
    matched_cxcywh = box_convert(matched_boxes_xyxy, 'xyxy', 'cxcywh')
    g_cxcy = (matched_cxcywh[:, :2] - anchors[:, :2]) / (anchors[:, 2:] * variances[0])
    g_wh = torch.log(matched_cxcywh[:, 2:] / anchors[:, 2:] + 1e-8) / variances[1]
    return torch.cat([g_cxcy, g_wh], dim=1)


def decode_boxes(loc_preds, variances=(0.1, 0.2)):
    """ネットワークが出力したオフセットを，画像上のボックス座標（xyxy, ピクセル座標）にデコードする"""
    cxcy = loc_preds[..., :2] * variances[0] * anchors[:, 2:] + anchors[:, :2]
    wh = torch.exp(loc_preds[..., 2:] * variances[1]) * anchors[:, 2:]
    boxes_cxcywh = torch.cat([cxcy, wh], dim=-1)
    boxes_xyxy = box_convert(boxes_cxcywh, 'cxcywh', 'xyxy')
    return boxes_xyxy.clamp(min=0, max=IMG_SIZE)  # 画像範囲外に出たボックスをクリップする


def match_anchors(gt_boxes, gt_labels, pos_iou=0.5, neg_iou=0.4):
    """1枚の画像内のGTボックスをアンカーに割り当て，学習用のラベル・オフセットターゲットを作成する"""
    num_anchors_total = anchors.shape[0]
    if gt_boxes.shape[0] == 0:
        labels = torch.zeros(num_anchors_total, dtype=torch.long, device=device)  # GTなし -> 全て背景
        loc_targets = torch.zeros(num_anchors_total, 4, device=device)
        return labels, loc_targets

    ious = box_iou(gt_boxes, anchors_xyxy)  # (num_gt, num_anchors)

    # 各アンカーに対して，最もIoUが高いGTを割り当てる
    best_gt_iou, best_gt_idx = ious.max(dim=0)  # (num_anchors,)
    # 各GTボックスに対して，最もIoUが高いアンカーは，閾値によらず必ず正例にする
    _, best_anchor_idx = ious.max(dim=1)  # (num_gt,)
    best_gt_iou[best_anchor_idx] = 1.0
    best_gt_idx[best_anchor_idx] = torch.arange(gt_boxes.shape[0], device=gt_boxes.device)

    labels = torch.full((num_anchors_total,), -1, dtype=torch.long, device=device)  # デフォルトは学習対象外
    labels[best_gt_iou < neg_iou] = 0  # 背景
    pos_mask = best_gt_iou >= pos_iou
    labels[pos_mask] = gt_labels[best_gt_idx[pos_mask]]  # 1〜20のVOCクラスラベル

    matched_boxes = gt_boxes[best_gt_idx]
    loc_targets = encode_boxes(matched_boxes)

    return labels, loc_targets

## 損失関数（Focal Loss + Smooth L1）
本ノートブックのアンカーは大部分が背景であり，これは`ssd.ipynb`のデフォルトボックスと同じ状況である．SSDではHard Negative Miningで対処したが，EfficientDet（RetinaNet）では**Focal Loss**によってこの不均衡そのものを損失関数側で解決する．

Focal Lossは，クロスエントロピー損失に「正しく分類できている（$p_t$が大きい）サンプルの寄与を小さくする」係数$(1-p_t)^\gamma$を掛けたものである．

$$FL(p_t) = -\alpha_t (1 - p_t)^{\gamma} \log(p_t)$$

背景クラスのように容易に分類できる大量の負例の損失を自動的に小さく抑えるため，Hard Negative Miningのような明示的なサンプリングをしなくても，少数の正例（物体）に学習を集中させることができる．$\gamma \approx 2.0$，$\alpha \approx 0.25$を用いる．

また，Focal Lossは各クラスを独立した2値分類（Sigmoid）として扱う設計が一般的である．そのため本ノートブックでは，`ssd.ipynb`のような「背景を含めた21クラスのSoftmax」ではなく，**背景クラスを明示的には持たない20クラスのSigmoid**という設計にする（正例アンカーは対応するクラスだけが1，それ以外の19クラスは0の多ラベル的なターゲットとし，背景アンカーは20クラス全てが0のターゲットとなる）．ボックス回帰は，`ssd.ipynb`と同様に正例アンカーのみを対象としたSmooth L1損失を用いる．

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, cls_preds, labels):
        # labels: (num_valid_anchors,) 学習対象外（-1）のアンカーは事前に除外して渡される
        n_class = cls_preds.shape[-1]
        targets = torch.zeros_like(cls_preds)
        pos_mask = labels > 0
        targets[pos_mask, labels[pos_mask] - 1] = 1.0  # 1〜20のラベルを0〜19のクラスIndexに変換してone-hot化

        p = torch.sigmoid(cls_preds)
        pt = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = -alpha_t * (1 - pt).pow(self.gamma) * torch.log(pt.clamp(min=1e-8))

        num_pos = pos_mask.sum().clamp(min=1)
        return loss.sum() / num_pos


class DetectionLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.focal_loss = FocalLoss(alpha, gamma)
        self.smooth_l1 = nn.SmoothL1Loss(reduction='sum')

    def forward(self, cls_preds, box_preds, labels, loc_targets):
        valid_mask = labels != -1  # IoUが中途半端なアンカーは損失計算から除外する

        cls_loss = self.focal_loss(cls_preds[valid_mask], labels[valid_mask])

        pos_mask = labels > 0
        num_pos = pos_mask.sum().clamp(min=1)
        box_loss = self.smooth_l1(box_preds[pos_mask], loc_targets[pos_mask]) / num_pos

        return cls_loss + box_loss

## ネットワークの作成
バックボーン（EfficientNet-B0）・BiFPN（3回繰り返し，チャンネル数64）・検出ヘッドを組み合わせて`EfficientDet`モデルを構築する．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.001，モーメンタム0.9，weight decay 5e-4）を用いる．事前学習済みのバックボーンを使用するため，スクラッチ学習のCIFAR100分類モデルよりも小さめの学習率を設定している．

In [ ]:
class EfficientDet(nn.Module):
    def __init__(self, n_class, num_anchors, channels=64, bifpn_repeats=3):
        super().__init__()
        self.backbone = EfficientDetBackbone()
        self.bifpn = BiFPN(self.backbone.out_channels, channels=channels, num_repeats=bifpn_repeats)
        self.head = DetectionHead(channels, num_anchors, n_class)

    def forward(self, x):
        c3, c4, c5 = self.backbone(x)
        p3, p4, p5 = self.bifpn(c3, c4, c5)
        return self.head([p3, p4, p5])


model = EfficientDet(n_class=n_class, num_anchors=num_anchors_per_cell)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
criterion = DetectionLoss(alpha=0.25, gamma=2.0)

n_params = sum(p.numel() for p in model.parameters())
print('総パラメータ数:', n_params)

## 学習
読み込んだVOC2007データセットと作成したネットワークを用いて学習を行う．ミニバッチサイズを8，学習エポック数を20とする．画像1枚あたりの物体数が異なるため，`collate_fn`でバッチを作成し，各画像のGTボックスをバッチ内でループしながらアンカーとマッチングさせる．

In [ ]:
batch_size = 8
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate_fn)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for images, targets in train_loader:
        images = images.to(device)

        batch_labels, batch_loc_targets = [], []
        for t in targets:
            labels, loc_targets = match_anchors(t['boxes'].to(device), t['labels'].to(device))
            batch_labels.append(labels)
            batch_loc_targets.append(loc_targets)
        batch_labels = torch.stack(batch_labels)
        batch_loc_targets = torch.stack(batch_loc_targets)

        cls_preds, box_preds = model(images)

        loss = criterion(cls_preds, box_preds, batch_labels, batch_loc_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print("epoch: {}, mean loss: {}, elapsed time: {}".format(epoch, sum_loss / len(train_loader), time() - train_start))

## 推論（デコード・NMS）
学習したモデルで推論を行うための関数を定義する．出力されたクラスごとのロジットにSigmoidを適用し，クラスごとに信頼度が閾値を超えるアンカーをデコードしたのち，`torchvision.ops.nms`でクラスごとに重複した検出結果を除去（Non-Maximum Suppression）する．`ssd.ipynb`との違いは，Softmaxではなくクラスごとに独立したSigmoidスコアを使う点である．

In [ ]:
def predict(model, image_tensor, conf_threshold=0.3, nms_threshold=0.45):
    model.eval()
    with torch.no_grad():
        cls_preds, box_preds = model(image_tensor.unsqueeze(0).to(device))
    scores_all = torch.sigmoid(cls_preds[0])  # (num_anchors, n_class)
    boxes_all = decode_boxes(box_preds[0])

    result_boxes, result_scores, result_labels = [], [], []
    for class_idx in range(n_class):  # 0〜19（背景クラスは持たない）
        class_scores = scores_all[:, class_idx]
        mask = class_scores > conf_threshold
        if mask.sum() == 0:
            continue
        class_boxes = boxes_all[mask]
        class_scores = class_scores[mask]

        keep = nms(class_boxes, class_scores, nms_threshold)
        result_boxes.append(class_boxes[keep])
        result_scores.append(class_scores[keep])
        result_labels.append(torch.full((len(keep),), class_idx + 1, dtype=torch.long))  # GTと同じ1〜20の表現に合わせる

    if len(result_boxes) == 0:
        return torch.zeros(0, 4), torch.zeros(0), torch.zeros(0, dtype=torch.long)

    return torch.cat(result_boxes).cpu(), torch.cat(result_scores).cpu(), torch.cat(result_labels).cpu()

## テスト
評価指標には，物体検出タスクで広く使われる**mAP（mean Average Precision）**を用いる．`ssd.ipynb`と同様，算出には`torchmetrics`の`MeanAveragePrecision`を利用し，VOCで伝統的に使われるIoU閾値0.5でのmAP（`map_50`）を確認する．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

metric = MeanAveragePrecision(iou_type='bbox')
model.eval()

for images, targets in test_loader:
    boxes, scores, labels = predict(model, images[0])

    preds = [{'boxes': boxes, 'scores': scores, 'labels': labels}]
    gts = [{'boxes': targets[0]['boxes'], 'labels': targets[0]['labels']}]
    metric.update(preds, gts)

result = metric.compute()
print('mAP@0.5:', result['map_50'].item())
print('mAP@[0.5:0.95]:', result['map'].item())

## 検出結果の可視化
テストデータ1枚に対する検出結果を，`torchvision.utils.draw_bounding_boxes`を用いて画像上に描画する．

In [ ]:
image, target = test_data[0]
boxes, scores, labels = predict(model, image)

# 正規化を戻して0-255のuint8画像に変換
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
img_show = (image * std + mean).clamp(0, 1)
img_show = (img_show * 255).to(torch.uint8)

label_names = [f'{VOC_CLASSES[l - 1]}: {s:.2f}' for l, s in zip(labels, scores)]
img_with_boxes = draw_bounding_boxes(img_show, boxes, labels=label_names, colors='red', width=2)

plt.figure(figsize=(6, 6))
plt.imshow(img_with_boxes.permute(1, 2, 0))
plt.axis('off')
plt.show()

## オリジナルEfficientDetとの違い
本ノートブックで実装したEfficientDetは，特にBiFPNの学習可能な重み付き特徴融合という核心部分を再現しつつも，レクチャー用に構成を簡略化している．主な違いは次の通りである．

| | 原論文（EfficientDet-D0） | 本ノートブック |
|---|---|---|
| バックボーン | EfficientNet-B0（事前学習済み） | 同じ（本シリーズの他ノートブックの一般方針であるResNet50からの意図的な例外） |
| 入力画像サイズ | 512×512 | 512×512（変更なし） |
| 特徴ピラミッドの段数 | P3〜P7（5段階） | P3〜P5（3段階，実装を簡略化） |
| BiFPNのチャンネル数 | 64 | 64（変更なし） |
| BiFPNの繰り返し回数 | モデルサイズ（D0〜D7）に応じてcompound scalingで決定 | 3回に固定 |
| 重み付き特徴融合 | Fast Normalized Fusion | 同じ（`WeightedFusion`として実装） |
| 検出ヘッドのBatchNorm | ピラミッドレベルごとに個別のBN（畳み込み重みのみ共有） | 全レベルで共有した単一のBN（実装を簡略化） |
| クラス分類の設計 | 背景クラスを明示的に持たず，クラスごとに独立したSigmoidで予測 | 同じ |
| アンカーの割り当て | IoU閾値によるpositive/negative/ignoreの3段階 | 同じ |
| NMS・IoU計算 | 独自実装 | `torchvision.ops`の`nms`・`box_iou`を使用 |

特徴ピラミッドをP3〜P5の3段階に絞ったことにより，大きな物体（P6・P7が主に担当する）の検出精度は原論文のD0より劣ると考えられるが，BiFPNの双方向経路・重み付き融合というアーキテクチャ上の核心は変更せずに実装している．

## 課題

1. `BiFPN`の`num_repeats`（繰り返し回数）を変更し，パラメータ数・学習にかかる時間・mAPへの影響を確認しましょう．

2. 学習後に各`WeightedFusion`モジュールの`weights`（`F.relu`を通す前の値）を出力し，どの入力が強く重み付けされているかを確認しましょう．トップダウン経路とボトムアップ経路で傾向に違いがあるかも比較してみましょう．

3. `WeightedFusion`を，学習可能な重みを使わない単純な等重み和（あるいは平均）に置き換えたモデルを作成し，学習の収束の速さやmAPを比較してみましょう．重み付き融合がどの程度性能に寄与しているかを確認できます．